# KGC Adaptation Thesis — Kaggle runner

**Settings:** Accelerator **GPU T4 ×2** · Internet **ON** · Persistence **Variables and Files**

> **Nothing to attach.** Section 1 downloads all four KGs from KG-LLM's repo.

---

### ★ Three rules that caused every failure so far

**1 · Pin every GPU job to ONE device.** With two visible, HF Trainer silently wraps the model in `DataParallel`, autocast never reaches the replicas, and the fp32 adapter meets the fp16 base:

```
Caught RuntimeError in replica 0 on device 0
RuntimeError: mat1 and mat2 must have the same dtype, but got Float and Half
```

Two T4s are for **two independent jobs** (`pair()` below), never for splitting one.

**2 · `attn_implementation` is a CORRECTNESS setting, not a speed one.** Measured on this exact hardware by the dtype probe in section 2:

| dtype | attn | finite | max\|logit\| | VRAM |
|---|---|---|---|---|
| fp16 | eager | **False** | nan | 3.11 GB |
| fp16 | **sdpa** | True | 27.2 | **3.12 GB** |
| fp32 | eager | True | 27.3 | 6.22 GB |
| fp32 | sdpa | True | 27.3 | 6.23 GB |

`eager` in fp16 returns **NaN**, which surfaces as `train_loss=0.0` with `grad_norm=nan` and looks like a finished run. `configs/base.yaml` uses fp16 + sdpa.

**3 · MoRA and BOFT cannot coexist.** `peft-mora` is a fork of peft 0.9.0, which predates BOFT, and installing it **overwrites** official peft. One environment per session:

| `ENV` | installs | runnable |
|---|---|---|
| `"mora"` | the peft-mora fork | `lora`, `mora`, `probe`, `dpo` |
| `"official"` | latest official peft | `lora`, `boft`, `probe`, `dpo` |

**Run `--peft lora` in BOTH**, then `scripts/verify_env_control.py`. If the two LoRA numbers agree, peft version is not a confound and the three-way comparison stands.

> ⚠️ For training use **Save & Run All (Commit)** — interactive sessions die on disconnect.

## 0 · Clone + install

In [ ]:
# Internet is OFF by default in a fresh Kaggle notebook, and every failure
# downstream then looks like something else.
import socket, urllib.request

for host in ("github.com", "huggingface.co"):
    try:
        print(f"DNS   ok   {host} -> {socket.gethostbyname(host)}")
    except Exception as e:
        print(f"DNS   FAIL {host}: {e}    <-- Settings > Internet > ON")
try:
    print("HTTPS ok   status", urllib.request.urlopen("https://github.com", timeout=10).status)
except Exception as e:
    print("HTTPS FAIL:", e)

In [ ]:
REPO_URL = "https://github.com/lynda-lagh/contribution-.git"
PRIVATE  = False          # True only if the repo is private (needs a GITHUB_TOKEN secret)
DEST     = "/kaggle/working/repo"

import os, subprocess, sys

url = REPO_URL
if PRIVATE:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"$ {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

if os.path.isdir(f"{DEST}/.git"):
    # fetch + reset, NOT pull: once a run has written into the working dir,
    # `git pull` refuses to merge and you silently keep executing stale code.
    run(["git", "-C", DEST, "fetch", "--all"])
    run(["git", "-C", DEST, "reset", "--hard", "origin/main"])
    print("updated existing clone")
else:
    run(["git", "clone", "--depth", "1", url, DEST])
    print("cloned")

os.chdir(DEST)
if DEST not in sys.path:
    sys.path.insert(0, DEST)

print("HEAD:", run(["git", "log", "-1", "--oneline"]))
print("cwd :", os.getcwd())
print("\n\u2605 Does that hash match your latest push? If not, you forgot")
print("  `git push` on your PC and are about to re-run old code.")

In [ ]:
# \u2605 PICK ONE - see rule 3 at the top.
ENV = "mora"        # "mora" -> LoRA + MoRA    |    "official" -> LoRA + BOFT

!pip install -q -r requirements.txt

if ENV == "mora":
    # a FORK of peft 0.9.0 - installed LAST, because it overwrites official peft
    !pip install -q git+https://github.com/kongds/MoRA.git#subdirectory=peft-mora
elif ENV == "official":
    !pip install -q -U peft
else:
    raise ValueError("ENV must be 'mora' or 'official'")

import importlib, peft
importlib.reload(peft)
from src.utils.config import peft_env, usable_peft_methods
e = peft_env()
print(f"\nENV={ENV} | peft {peft.__version__} | environment detected: {e['peft_env']}")
print("runnable here:", ", ".join(usable_peft_methods()))

## 1 · Data — downloaded, not uploaded

Pulls the four files per dataset from [`yao8839836/kg-llm`](https://github.com/yao8839836/kg-llm/tree/main/data), then checks each exists, is non-empty, and that `test.tsv` carries the ±1 label in column 4. Descriptions are KG-BERT's — the field standard — so our numbers stay commensurable with the published ones.

In [ ]:
!python -m scripts.fetch_data --datasets WN11 FB13

In [ ]:
# Chapter 1 uses WN11 + FB13 only. Chapter 2 needs the big graphs --
# YAGO3-10 is ~1M triples, so there is no reason to pull it today.
FETCH_BIG = False

if FETCH_BIG:
    !python -m scripts.fetch_data --datasets WN18RR YAGO3-10
else:
    print("skipped WN18RR + YAGO3-10 -- set FETCH_BIG = True before Chapter 2\n")

# re-checks what is on disk; does not re-download
!python -m scripts.fetch_data --verify-only --datasets WN11 FB13

## 2 · Smoke test — ~5 min

It prints which **session** you are in and the run plan that belongs to it.

* `ENV="mora"` → LoRA and MoRA pass, **BOFT fails** and the script **exits 0**. That failure is the fork conflict, not your code.
* `ENV="official"` → LoRA and BOFT pass, **MoRA fails**. Mirrored.

⚠️ `train_loss` of exactly `0.0000` is a **failure**, not a pass. The test asserts the loss is strictly positive and that no trainable parameter is still fp16.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m scripts.smoke_test

## 3 · Build instruction data

10,000 triples → ~20,000 instances (1 positive + 1 negative each). Stratified by relation, min 10 per relation, so rare relations survive. No GPU needed.

`--anonymise` is the contamination control: entity surface forms replaced by opaque ids. WN11 and FB13 date from 2013 and predate every LLM's training cutoff, so it is not optional.

In [ ]:
!python -m src.data.build_instructions --dataset WN11 --n_triples 10000 --seed 42
!python -m src.data.build_instructions --dataset WN11 --n_triples 10000 --seed 42 --anonymise

## 4 · Chapter 1 — format vs knowledge

**Two training runs**, independent, so one per T4. Everything after is inference over the same generations.

Binary task → chance = 50%, which is what makes KG-LLM's untuned 21.1 / 9.1 unambiguous: a model cannot be wrong about facts five times more reliably than a coin.

> Evaluation uses a fixed 2,000-item subset of WN11's 21,088 test rows — identical items across every condition, which is what makes the paired significance tests valid. Say so in the paper; it means these numbers are not directly comparable to KG-LLM's full-test figures.

In [ ]:
import subprocess

def pair(a, b):
    """Two INDEPENDENT jobs, one per T4. Never DataParallel."""
    pa = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=0 {a}", shell=True)
    pb = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=1 {b}", shell=True)
    return pa.wait(), pb.wait()

pair("python -m chapters.ch1_diagnostic.run --dataset WN11",
     "python -m chapters.ch1_diagnostic.run --dataset WN11 --anonymise")

In [ ]:
# four parsers on identical outputs + SMI as a second, independent instrument
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch1_diagnostic.analyse --dataset WN11 --smi

## 5 · Chapter 2 — the |E| sweep

Set `FETCH_BIG = True` in section 1 first, and make sure `ENV` matches the `--peft` you are about to run.

In [ ]:
!python -m chapters.ch2_adaptation.run --sweep

In [ ]:
BASE = "python -m chapters.ch2_adaptation.run --dataset YAGO3-10 --triples 10000"

# ---- SESSION B: requires ENV = "mora" -------------------------------------
for E in (10000, 25000, 50000, 123182):
    print(f"===== |E| = {E:,} =====")
    pair(f"{BASE} --peft lora --entities {E}", f"{BASE} --peft mora --entities {E}")

In [ ]:
# ---- SESSION A: requires ENV = "official" ---------------------------------
# Restart the session, set ENV = "official", re-run section 0, then this cell.
#
# probe = frozen linear classifier on hidden states, no training. It is the
# CONTROL that makes a flat MoRA result interpretable rather than ambiguous:
# if every method merely matches a probe, no method installed knowledge.
pair(f"{BASE} --peft boft  --entities 123182",
     f"{BASE} --peft probe --entities 123182")

# \u2605 the cross-environment control: the SAME LoRA config as session B
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch2_adaptation.run --dataset YAGO3-10 --triples 10000 --peft lora --entities 123182

In [ ]:
# \u2605 Run AFTER both sessions. Compares the LoRA arm across peft environments.
# Set --tolerance from your own seed variance (ch2 analyse prints it), not from
# the default -- picking a tolerance that happens to pass is what an examiner
# will probe first.
!python -m scripts.verify_env_control --tolerance 0.01

### 5b · DPO — Phase 2, on the winner only

KG-LLM trains on `random.choice(all_entities)` — uniformly random, therefore usually type-violating and trivially separable. We replace those with type-consistent near-misses and ask whether preference optimisation reduces structural hallucination.

Verified 0 of 188 papers apply preference optimisation to KGC.

In [ ]:
SFT = "checkpoints/ch2-mora-E123182-T10000-s42"     # <-- the Phase-1 winner
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch2_adaptation.run --peft dpo --sft-adapter {SFT} \
        --negatives type_consistent --entities 123182 --triples 10000

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch2_adaptation.analyse --forgetting

## 6 · Chapter 3 — the conditioning ladder

Routing analysis + faithfulness cost **no training**. Run `--analyse` first and check the `rich` band: if it is ~0%, the ∅ branch never fires, and a 0% skip rate is a data problem rather than a result.

⚠️ Relation descriptions currently come from a template derived from the relation label. Fine for a smoke run — for the real Chapter 3 they must be LLM-generated, or L1 injects boilerplate and the first rung measures nothing.

In [ ]:
!python -m chapters.ch3_conditioning.run --dataset YAGO3-10 --analyse

In [ ]:
for LEVEL in ["L0", "L1", "L2", "L3"]:          # L4 is the first thing to cut
    !CUDA_VISIBLE_DEVICES=0 python -m chapters.ch3_conditioning.run --dataset YAGO3-10 --level {LEVEL} --train

## 7 · Chapter 4 — measurement

★ **ZERO training.** Inference over checkpoints from Chapters 2–3, which is why this chapter is the safest and holds the verified gaps: calibration (2 of 188), abstention (0), explanation faithfulness (0).

In [ ]:
ADAPTER = "checkpoints/ch2-mora-E123182-T10000-s42"
!CUDA_VISIBLE_DEVICES=0 python -m chapters.ch4_measurement.run --adapter {ADAPTER} --dataset YAGO3-10 --limit 2000

## 8 · Package results

Download `results.zip` from the notebook **Output** tab, then run the review app locally.

In [ ]:
!zip -qr /kaggle/working/results.zip results/
!du -sh /kaggle/working/results.zip
!ls -la checkpoints/ 2>/dev/null | head -20

---
### Session notes

* **Push before every session.** The clone cell prints the commit hash — check it.
* **This notebook lives in Kaggle, not in the repo.** Edits to `notebooks/kaggle_runner.ipynb` do **not** reach it; re-upload when it changes.
* **One GPU per job** — `CUDA_VISIBLE_DEVICES=0` / `=1`, never both.
* **12-hour cap** — checkpointing is on (`save_steps: 250`); resume from `checkpoints/<run>/checkpoint-*`
* **~30 GPU-h/week**, roughly doubled by the two-job pattern
* **Adapters are 20–100 MB** — safe for `/kaggle/working`
* **Download `results/` every session** — `/kaggle/working` is not permanent
* **Peak VRAM so far:** LoRA 3.28 GB · MoRA 6.21 GB (fp16 + sdpa, micro-batch 2)